In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

import re, time, unicodedata, json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# ── Pilih backend ────────────────────────────────────────
BACKEND = 'groq'

# Groq (cloud)
GROQ_API_KEY = GROQ_API_KEY.strip()
GROQ_MODEL = "llama-3.3-70b-versatile"

# ── Context window ───────────────────────────────────────
MAX_CHARS = 3500  # chunk size for zero-shot-style truncation
MAX_CHARS_FEW_SHOT = 2400  # smaller chunk size when using few-shot examples

# Data Paths
DATA_PATH = Path('../../datasets')
PROCESSED_PATH = DATA_PATH / 'processed_data/processed.pkl'
VOCABS_PATH = DATA_PATH / 'processed_data/vocabs.pkl'
SPLITS_PATH = DATA_PATH / 'split_data/split_indices.json'

print('Config loaded OK')
print(f'Backend : {BACKEND} | Model: {GROQ_MODEL}')


Config loaded OK
Backend : groq | Model: llama-3.3-70b-versatile


In [2]:
with open(PROCESSED_PATH, 'rb') as f:
    processed = pickle.load(f)
with open(VOCABS_PATH, 'rb') as f:
    vocabs = pickle.load(f)
with open(SPLITS_PATH, 'r', encoding='utf-8') as f:
    splits = json.load(f)

idx_train = splits['idx_train']
idx_val = splits['idx_val']
idx_test = splits['idx_test']

ENTITY_LABELS = vocabs['Entity_labels'] + ['UNKNOWN']
id2labels = vocabs['id2label']

print(f'Total records: {len(idx_train)} / {len(idx_val)} / {len(idx_test)}')
print(f'Entity labels: {ENTITY_LABELS}')
print(f'Example test record {idx_test[0]} length: {len(processed[idx_test[0]]["content"])} chars')


Total records: 154 / 33 / 33
Entity labels: ['Name', 'Designation', 'Companies worked at', 'Location', 'Email Address', 'College Name', 'Degree', 'Graduation Year', 'Skills', 'Years of Experience', 'UNKNOWN']
Example test record 132 length: 8706 chars


In [3]:
def extract_ground_truth(record: dict) -> dict:
    gt = defaultdict(list)
    content = record['content']

    for start, end, label in record['spans']:
        text = content[start:end].strip()
        if text and text not in gt[label]:
            gt[label].append(text)

    return dict(gt)


def clean_resume_text(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(f'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'[\u2022\u00b7\u25aa\u25cf\u25a0\u25e6\u25cb\u25b8\u25ba]', '-', text)
    text = re.sub(r'[^\S\n]+', ' ', text)
    return text.strip()


def truncate_if_needed(text: str, max_chars: int = MAX_CHARS) -> list:
    if len(text) <= max_chars:
        return [text]

    chunks, current = [], ''
    for para in text.split('\n\n'):
        if len(current) + len(para) + 2 <= max_chars:
            current += para + '\n\n'
        else:
            if current:
                chunks.append(current.strip())
            current = para + '\n\n'

    if current.strip():
        chunks.append(current.strip())

    return chunks if chunks else [text[:max_chars]]


lengths = [len(processed[i]['content']) for i in idx_test]
print(f'Test set resume length: min: {min(lengths)} max: {max(lengths)} mean: {sum(lengths) / len(lengths):.1f}')
print(f'Resume > MAX_CHARS ({MAX_CHARS}): {sum(1 for l in lengths if l > MAX_CHARS)}')


Test set resume length: min: 388 max: 12267 mean: 3800.1
Resume > MAX_CHARS (3500): 14


In [4]:
system_prompt = """You are an expert Named Entity Recognition (NER) system specialized in extracting structured information from resumes and professional profiles.

You MUST follow these rules strictly:
1. Extract ONLY entities explicitly present in the text. Do NOT infer or hallucinate information.
2. Assign exactly ONE label per entity from the allowed label set.
3. If an entity does not clearly belong to any defined label, assign the label UNKNOWN.
4. Return your response as a valid JSON object. No extra explanation, no markdown, no preamble.
5. Each key in the JSON must map to a list of unique string values.
6. Normalize values: strip extra whitespace, proper nouns capitalized.

ALLOWED LABELS:
- Company worked at → Name of a company or organization the person worked at
- Designation → Job title or role
- Skills → Technical skill, tool, language, framework, or soft skill
- Location → City, state, country, or geographic region
- College Name → Name of a university, college, or educational institution
- Degree → Academic degree or qualification
- Graduation Year → Year of graduation (4-digit year)
- Email Address → Valid email address
- Name → Full name of the candidate
- Years of Experience → Duration of professional experience
- UNKNOWN → Entity present but does not fit any label above

OUTPUT FORMAT (strict JSON):
{
  "Name": [], "Email Address": [], "Years of Experience": [], "Company worked at": [],
  "Designation": [], "Skills": [], "Location": [], "College Name": [],
  "Degree": [], "Graduation Year": [], "UNKNOWN": []
}"""

FEW_SHOT_K = 2
FEW_SHOT_MAX_LEN = 1200


def select_few_shot_examples(processed, idx_candidates, k=FEW_SHOT_K, max_len=FEW_SHOT_MAX_LEN):
    examples = []
    for idx in idx_candidates:
        record = processed[idx]
        text = clean_resume_text(record['content'])
        if len(text) <= max_len:
            examples.append(record)
            if len(examples) == k:
                break
    if len(examples) < k:
        examples = [processed[i] for i in idx_candidates[:k]]
    return examples


def build_prompt_messages(resume_text: str, examples: list) -> list:
    messages = [{'role': 'system', 'content': system_prompt}]

    for example in examples:
        example_text = clean_resume_text(example['content'])
        example_gt = extract_ground_truth(example)

        messages.append({
            'role': 'user',
            'content': (
                'Extract entities. Return ONLY JSON.\n\n'
                'RESUME:\n"""\n' + example_text + '\n"""'
            )
        })
        messages.append({
            'role': 'assistant',
            'content': json.dumps(example_gt, indent=2)
        })

    messages.append({
        'role': 'user',
        'content': (
            'Extract entities. Return ONLY JSON.\n\n'
            'RESUME:\n"""\n' + resume_text + '\n"""'
        )
    })

    return messages


few_shot_examples = select_few_shot_examples(processed, idx_train)
print('Few-shot examples selected:')
for i, record in zip(idx_train, few_shot_examples):
    print('-', i, 'length', len(record['content']))


Few-shot examples selected:
- 147 length 1105
- 36 length 494


In [5]:
def parse_output(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', raw_text)
        try:
            result = json.loads(match.group()) if match else {}
        except Exception:
            result = {}

    for label in ENTITY_LABELS:
        result.setdefault(label, [])

    for key in list(result.keys()):
        if not isinstance(result[key], list):
            result[key] = [str(result[key])] if result[key] else []
        result[key] = [str(v).strip() for v in result[key] if str(v).strip()]

    return result


def merge_chunks(results: list) -> dict:
    merged = defaultdict(set)
    for r in results:
        for label, vals in r.items():
            for v in vals:
                merged[label].add(v.lower().strip())
    return {l: sorted(list(v)) for l, v in merged.items()}


def call_groq_few_shot(messages: list, retries=2) -> str:
    import groq

    client = groq.Groq(api_key=GROQ_API_KEY)

    for i in range(retries + 1):
        try:
            r = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                temperature=0.0,
                max_tokens=1024
            )
            return r.choices[0].message.content
        except Exception as e:
            if i == retries:
                raise
            time.sleep(2 ** i)


def extract_entities_few_shot(resume_text: str) -> dict:
    cleaned = clean_resume_text(resume_text)
    chunks = truncate_if_needed(cleaned, max_chars=MAX_CHARS_FEW_SHOT)

    results = []
    for chunk in chunks:
        messages = build_prompt_messages(chunk, few_shot_examples)
        raw = call_groq_few_shot(messages)
        results.append(parse_output(raw))

    return merge_chunks(results) if len(chunks) > 1 else results[0]


In [6]:
def normalize_val(v: str) -> str:
    return re.sub(r'\s+', ' ', v.lower().strip())


def compute_metrics(pred: dict, gt: dict) -> dict:
    results = {}
    for label in ENTITY_LABELS:
        p = set(normalize_val(v) for v in pred.get(label, []))
        g = set(normalize_val(v) for v in gt.get(label, []))

        results[label] = {
            'tp': len(p & g),
            'fp': len(p - g),
            'fn': len(g - p)
        }
    return results


def aggregate_metrics(all_results: list) -> pd.DataFrame:
    totals = {l: {'tp': 0, 'fp': 0, 'fn': 0} for l in ENTITY_LABELS}

    for rec in all_results:
        for l in ENTITY_LABELS:
            for k in ('tp', 'fp', 'fn'):
                totals[l][k] += rec.get(l, {}).get(k, 0)

    rows = []
    for l in ENTITY_LABELS:
        tp = totals[l]['tp']
        fp = totals[l]['fp']
        fn = totals[l]['fn']
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        rows.append({
            'Label': l,
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'Precision': round(p, 3),
            'Recall': round(r, 3),
            'F1': round(f1, 3)
        })

    df = pd.DataFrame(rows)
    m = df[['Precision', 'Recall', 'F1']].mean()
    df = pd.concat([
        df,
        pd.DataFrame([{
            'Label': 'MACRO AVG',
            'TP': '',
            'FP': '',
            'FN': '',
            'Precision': round(m['Precision'], 3),
            'Recall': round(m['Recall'], 3),
            'F1': round(m['F1'], 3)
        }])
    ], ignore_index=True)

    return df

print('Evaluation functions loaded OK')


Evaluation functions loaded OK


In [7]:
all_metrics = []
predictions = []
errors = []

for resume_idx in tqdm(idx_test, desc='Test set'):
    record = processed[resume_idx]
    gt = extract_ground_truth(record)
    try:
        pred = extract_entities_few_shot(record['content'])
        metrics = compute_metrics(pred, gt)

        all_metrics.append(metrics)
        predictions.append({'resume_idx': resume_idx, 'pred': pred, 'gt': gt})
    except Exception as e:
        print(f'Error idx {resume_idx}: {e}')
        errors.append(resume_idx)
        all_metrics.append({})

print(f'\nDone. Test: {len(idx_test)} | Errors: {len(errors)}')


df = aggregate_metrics(all_metrics)
print(f'=== Few-Shot Llama NER — Test Set ({len(idx_test)} samples) ===')
print(df.to_string(index=False))

try:
    display(
        df.style
        .highlight_max(subset=['F1'], color='lightgreen')
        .highlight_min(subset=['F1'], color="#c35757")
        .set_caption(
            f'Few-Shot Llama — Test Set ({len(idx_test)} samples)'
        )
    )
except Exception:
    pass


Test set:  85%|████████▍ | 28/33 [07:29<00:37,  7.52s/it]

Error idx 16: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk4ny2bfc3vnzqg0qsv08fc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99579, Requested 1876. Please try again in 20m57.12s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Test set:  88%|████████▊ | 29/33 [07:33<00:25,  6.37s/it]

Error idx 45: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk4ny2bfc3vnzqg0qsv08fc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99575, Requested 1953. Please try again in 22m0.192s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Test set:  91%|█████████ | 30/33 [07:37<00:16,  5.59s/it]

...

Test set:  97%|█████████▋| 32/33 [07:44<00:04,  4.69s/it]

Error idx 55: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk4ny2bfc3vnzqg0qsv08fc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99561, Requested 1928. Please try again in 21m26.496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Test set: 100%|██████████| 33/33 [07:48<00:00, 14.20s/it]

Error idx 108: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk4ny2bfc3vnzqg0qsv08fc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99557, Requested 1919. Please try again in 21m15.264s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Done. Test: 33 | Errors: 6
=== Few-Shot Llama NER — Test Set (33 samples) ===
              Label TP  FP FN  Precision  Recall    F1
               Name 25   4  2      0.862   0.926 0.893
        Designation 22  40  7      0.355   0.759 0.484
Companies worked at 21  26  3      0.447   0.875 0.592
           Location 23  63  2      0.267   0.920 0.414
      Email Address 14  13  9      0.519   0.609 0.560
       College Name 21  22  7      0.488   0.750 0.592
             Degree 19  23 11      0.452   0.633 0.528
    Graduation Year 21  14  2      0.600   0.913 0.724
 

,Label,TP,FP,FN,Precision,Recall,F1
0,Name,25,4,2,0.862000,0.926000,0.893000
1,Designation,22,40,7,0.355000,0.759000,0.484000
2,Companies worked at,21,26,3,0.447000,0.875000,0.592000
3,Location,23,63,2,0.267000,0.920000,0.414000
4,Email Address,14,13,9,0.519000,0.609000,0.560000
5,College Name,21,22,7,0.488000,0.750000,0.592000
6,Degree,19,23,11,0.452000,0.633000,0.528000
7,Graduation Year,21,14,2,0.600000,0.913000,0.724000
8,Skills,10,512,28,0.019000,0.263000,0.036000
9,Years of Experience,1,10,1,0.091000,0.500000,0.154000
